In [1]:
!pip install requests


In [2]:
import pandas as pd
import requests

# =========================================================
# STEP 1: Fetch Historical Weather Data from Open-Meteo
# =========================================================
print("Fetching weather data for Pune district...")

# Pune coordinates
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 18.5204,
    "longitude": 73.8567,
    "start_date": "2021-01-01",  # 1 month before price data starts
    "end_date": "2026-02-28",    # End of your dataset
    "daily": ["temperature_2m_mean", "precipitation_sum"],
    "timezone": "Asia/Kolkata"
}

response = requests.get(url, params=params)
weather_json = response.json()

# Create the initial weather dataframe
weather_df = pd.DataFrame({
    'arrival_date': pd.to_datetime(weather_json['daily']['time']),
    'temp_mean': weather_json['daily']['temperature_2m_mean'],
    'rainfall_daily': weather_json['daily']['precipitation_sum']
})

Fetching weather data for Pune district...


In [5]:
# =========================================================
# STEP 2: Weather Feature Engineering (Crucial Shift)
# =========================================================
print("Calculating and lagging weather features...")

# Ensure chronological order
weather_df = weather_df.sort_values('arrival_date')

# We shift by 1 to ensure we only use weather data up to YESTERDAY 
# to prevent real-time data leakage when predicting tomorrow.
weather_df['temp_mean_lag7'] = weather_df['temp_mean'].shift(7)
weather_df['rainfall_lag7'] = weather_df['rainfall_daily'].shift(7)

# 7-Day and 30-Day Rolling Features (Using the already shifted data)
weather_df['rainfall_7d_sum'] = weather_df['rainfall_lag7'].rolling(window=7).sum()
weather_df['rainfall_30d_sum'] = weather_df['rainfall_lag7'].rolling(window=30).sum()

weather_df['temp_7d_avg'] = weather_df['temp_mean_lag7'].rolling(window=7).mean()

# Drop the raw un-lagged columns so we don't accidentally use them in the model
weather_df = weather_df.drop(columns=['temp_mean', 'rainfall_daily'])

# Drop the initial NaNs created by the 30-day rolling window (Jan 2021)
weather_df = weather_df.dropna()

Calculating and lagging weather features...


In [7]:
# =========================================================
# STEP 3: Merge Weather with Master Price Dataset
# =========================================================
print("Loading master price dataset and merging...")
data_path = '../data/prepared_onion_pune_dynamic_master_7day.csv'
# Load the dataset you saved at the end of the previous pipeline
master_df = pd.read_csv(data_path)
master_df['arrival_date'] = pd.to_datetime(master_df['arrival_date'])

# Merge using a Left Join. 
# Left join ensures we don't lose any price rows; it just attaches the weather for that day.
final_df = pd.merge(master_df, weather_df, on='arrival_date', how='left')

# Check if any NaNs were introduced during the merge (should be 0 if dates align)
missing_weather = final_df['rainfall_7d_sum'].isna().sum()
print(f"Rows missing weather data after merge: {missing_weather}")

Loading master price dataset and merging...
Rows missing weather data after merge: 0


In [8]:
# =========================================================
# STEP 4: Final Cleanup and Save
# =========================================================
# If there are any slight API mismatches at the very end of the date range, forward fill them
weather_cols = ['temp_mean_lag7', 'rainfall_lag7', 'rainfall_7d_sum', 'rainfall_30d_sum', 'temp_7d_avg']
final_df[weather_cols] = final_df[weather_cols].ffill()

final_csv_name = 'final_model_ready_pune_data_7day.csv'
final_df.to_csv(final_csv_name, index=False)

print(f"Success! Final dataset saved to '{final_csv_name}' with {len(final_df)} records.")
print("Features included:", weather_cols)


Success! Final dataset saved to 'final_model_ready_pune_data_7day.csv' with 24811 records.
Features included: ['temp_mean_lag7', 'rainfall_lag7', 'rainfall_7d_sum', 'rainfall_30d_sum', 'temp_7d_avg']
